# NB12: OpenSlideFM internal TCGA evaluation

Evaluates OpenSlideFM patient embeddings on TCGA val/test using the same
PCA -> Ridge -> Platt pipeline as NB07. Reproduces the 0.812 AUROC test
performance reported in Table 3 of the manuscript.

**Manuscript reference** (Methods + Table 3 + Discussion):
- OpenSlideFM: ViT-L/14 backbone, 1,536-dimensional embeddings, dual-scale
  pretraining (0.5 + 2.0 um/px); see Sjtu-Fuxilab/OpenSlideFM repo
- Same PCA(384) -> Ridge(alpha=30.0) -> Platt pipeline as OpenCLIP
- TCGA test AUROC = 0.812 (delta = +0.046 vs OpenCLIP 0.766)
- Same train/val/test splits as NB07

OpenSlideFM feature extraction is performed in the OpenSlideFM repo
(typically `tcga_osfm_concat_0p5_2p0.parquet` containing patient-level
mean-pooled features). This notebook expects that parquet on disk.

**Required env**: `WORKSPACE`, `OSFM_FEATURES_PARQUET` (path to TCGA
patient-level OpenSlideFM embeddings, columns `patient` plus 1,536
feature columns).
**Outputs**: `models/frozen_model_osfm.joblib`,
`results/osfm_internal/{overall_metrics.csv, report.json}`.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# env
WORKSPACE = Path(os.environ.get("WORKSPACE", "./workspace")).resolve()
OSFM_FEATURES_PARQUET = Path(os.environ["OSFM_FEATURES_PARQUET"]).resolve()
LABELS_DIR = WORKSPACE / "labels"
MODELS_DIR = WORKSPACE / "models"
RESULTS_DIR = WORKSPACE / "results" / "osfm_internal"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# manuscript-locked params
SEED = 42
PCA_N = 384
RIDGE_ALPHA = 30.0
HRD_THR = 33
BOOT_N = 200
DROP_NAN_THR = 0.02         # drop columns with > 2% missing on train
CORR_BAN = 0.98             # leak-guard correlation threshold

print(f"OSFM_FEATURES_PARQUET: {OSFM_FEATURES_PARQUET}")
print(f"SEED={SEED}, PCA_N={PCA_N}, RIDGE_ALPHA={RIDGE_ALPHA}, HRD_THR={HRD_THR}, BOOT_N={BOOT_N}")


In [ ]:
# load OSFM features + labels, align
assert OSFM_FEATURES_PARQUET.exists(), f"missing {OSFM_FEATURES_PARQUET}; extract OSFM features first"
X = pd.read_parquet(OSFM_FEATURES_PARQUET)
if "patient" in X.columns:
    X = X.set_index("patient")
X.index = X.index.astype(str).str.upper().str.slice(0, 12)

labels = pd.read_parquet(LABELS_DIR / "labels.parquet")
labels["patient"] = labels["patient"].astype(str).str.upper().str.slice(0, 12)
labels = labels.drop_duplicates("patient").set_index("patient")

common = sorted(set(X.index) & set(labels.index))
X = X.loc[common]
labels = labels.loc[common].copy()
labels = labels.loc[labels["HRD"].notna()].copy()
X = X.loc[labels.index]
labels["HRD_top20"] = (labels["HRD"] >= HRD_THR).astype(int)

print(f"aligned patients with HRD: {len(labels):,}")
print(f"OSFM feature dim         : {X.shape[1]}  (manuscript ref: 1,536)")


In [ ]:
# split indices
idx_tr = labels.index[labels["split"] == "train"]
idx_va = labels.index[labels["split"] == "val"]
idx_te = labels.index[labels["split"] == "test"]
print(f"splits: train={len(idx_tr):,} val={len(idx_va):,} test={len(idx_te):,}")
print(f"manuscript ref: 6,465 / 811 / 833")


In [ ]:
# train-side sanitization: drop columns with >2% missing on TRAIN, impute median, drop near-constant
fcols = list(X.columns)
X_tr_arr = X.loc[idx_tr].values.astype(np.float64)

miss = np.mean(np.isnan(X_tr_arr), axis=0)
keep_step1 = miss <= DROP_NAN_THR
n_drop_missing = int((~keep_step1).sum())

X = X.loc[:, [c for c, k in zip(fcols, keep_step1) if k]]
fcols = list(X.columns)

imputer = SimpleImputer(strategy="median").fit(X.loc[idx_tr].values)
X_tr_i = imputer.transform(X.loc[idx_tr].values)

var = X_tr_i.var(axis=0)
keep_step2 = var > 1e-12
n_drop_const = int((~keep_step2).sum())

X = X.loc[:, [c for c, k in zip(fcols, keep_step2) if k]]
fcols = list(X.columns)

print(f"dropped >2% missing  : {n_drop_missing}")
print(f"dropped near-constant: {n_drop_const}")
print(f"surviving feature dim: {len(fcols)}")


In [ ]:
# leak-guard: drop columns with |Pearson r| >= 0.98 vs HRD_top20 on TRAIN
y_tr_bin = labels.loc[idx_tr, "HRD_top20"].astype(int).values
y_tr_cont = labels.loc[idx_tr, "HRD"].astype(float).values
X_tr_full = imputer.transform(X.loc[idx_tr].values)

def col_corr(arr, y):
    arr_z = (arr - arr.mean(axis=0)) / (arr.std(axis=0, ddof=0) + 1e-12)
    y_z = (y - y.mean()) / (y.std(ddof=0) + 1e-12)
    return arr_z.T @ y_z / len(y)

r_bin = col_corr(X_tr_full.astype(np.float64), y_tr_bin.astype(np.float64))
r_cont = col_corr(X_tr_full.astype(np.float64), y_tr_cont.astype(np.float64))
ban_mask = (np.abs(r_bin) >= CORR_BAN) | (np.abs(r_cont) >= CORR_BAN)
n_drop_corr = int(ban_mask.sum())

X = X.loc[:, [c for c, b in zip(fcols, ban_mask) if not b]]
fcols = list(X.columns)
print(f"max |r| with HRD_top20 on TRAIN: {np.nanmax(np.abs(r_bin)):.3f}")
print(f"max |r| with HRD continuous   : {np.nanmax(np.abs(r_cont)):.3f}")
print(f"corr-flagged dropped          : {n_drop_corr}")
print(f"final feature dim             : {len(fcols)}")


In [ ]:
# fit pipeline + Platt
X_tr = imputer.transform(X.loc[idx_tr].values).astype(np.float32)
X_va = imputer.transform(X.loc[idx_va].values).astype(np.float32)
X_te = imputer.transform(X.loc[idx_te].values).astype(np.float32)

pca_n = min(PCA_N, X_tr.shape[1] - 1, X_tr.shape[0] - 1)
pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("pca", PCA(n_components=pca_n, random_state=SEED)),
    ("ridge", Ridge(alpha=RIDGE_ALPHA, random_state=SEED)),
])
pipe.fit(X_tr, y_tr_cont)

z_tr = pipe.predict(X_tr).reshape(-1, 1)
platt = LogisticRegression(max_iter=1000, solver="lbfgs", random_state=SEED).fit(z_tr, y_tr_bin)

def predict(Xq):
    return platt.predict_proba(pipe.predict(Xq).reshape(-1, 1))[:, 1]

p_va = predict(X_va)
p_te = predict(X_te)
y_va = labels.loc[idx_va, "HRD_top20"].astype(int).values
y_te = labels.loc[idx_te, "HRD_top20"].astype(int).values


In [ ]:
# evaluate with bootstrap CIs
def boot_ci(y, p, fn, n_boot=BOOT_N, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y[idx], p[idx]
        if yb.sum() == 0 or yb.sum() == n:
            continue
        try:
            vals.append(fn(yb, pb))
        except Exception:
            pass
    if not vals:
        return (float("nan"), float("nan"))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def metrics(y, p):
    auc = float(roc_auc_score(y, p))
    ap = float(average_precision_score(y, p))
    br = float(brier_score_loss(y, p))
    auc_lo, auc_hi = boot_ci(y, p, roc_auc_score)
    ap_lo, ap_hi = boot_ci(y, p, average_precision_score)
    return {"AUROC": auc, "AUROC_lo": auc_lo, "AUROC_hi": auc_hi,
            "AP": ap, "AP_lo": ap_lo, "AP_hi": ap_hi, "Brier": br}

m_va = metrics(y_va, p_va)
m_te = metrics(y_te, p_te)
print(f"OSFM val  AUROC = {m_va['AUROC']:.3f} ({m_va['AUROC_lo']:.3f}-{m_va['AUROC_hi']:.3f})")
print(f"OSFM test AUROC = {m_te['AUROC']:.3f} ({m_te['AUROC_lo']:.3f}-{m_te['AUROC_hi']:.3f})  "
      f"  manuscript ref: 0.812")
print(f"delta vs OpenCLIP test (0.766): {m_te['AUROC'] - 0.766:+.3f}  "
      f"  manuscript ref: +0.046")

pd.DataFrame([{"split": "val", **m_va, "n": int(len(y_va)), "n_pos": int(y_va.sum())},
              {"split": "test", **m_te, "n": int(len(y_te)), "n_pos": int(y_te.sum())}]
            ).to_csv(RESULTS_DIR / "overall_metrics.csv", index=False)


In [ ]:
# save frozen OSFM model package, write report
joblib.dump({
    "pipe": pipe,
    "platt": platt,
    "imputer": imputer,
    "feature_columns": list(X.columns),
    "feat_dim": int(X.shape[1]),
    "pca_n": pca_n,
    "ridge_alpha": RIDGE_ALPHA,
    "threshold_HRD": HRD_THR,
    "seed": SEED,
    "backbone": "openslidefm_vitl14_dualscale",
}, MODELS_DIR / "frozen_model_osfm.joblib")

report = {
    "seed": SEED,
    "params": {"PCA_N": PCA_N, "RIDGE_ALPHA": RIDGE_ALPHA, "HRD_THR": HRD_THR, "BOOT_N": BOOT_N},
    "n_train": int(len(idx_tr)), "n_val": int(len(idx_va)), "n_test": int(len(idx_te)),
    "input_dim": int(X.shape[1]),
    "pca_n_used": int(pca_n),
    "metrics": {"val": m_va, "test": m_te},
    "manuscript_refs": {
        "test_AUROC": 0.812,
        "delta_vs_openclip": 0.046,
        "openclip_test_AUROC": 0.766,
        "feat_dim": 1536,
    },
    "frozen_model_path": str(MODELS_DIR / "frozen_model_osfm.joblib"),
}
(RESULTS_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))
